In [ ]:
import xscen as xs
import xarray as xr
from xscen import CONFIG
from matplotlib import pyplot as plt
import xclim as xc
import figanos.matplotlib as fg
from datetime import datetime
import cartopy.crs as ccrs
from matplotlib.colors import LogNorm
fg.utils.set_mpl_style('ouranos')
xs.load_config("../config/config_general.yml", "../config/config_region.yml", "../config/paths.yml")

/cvmfs/soft.computecanada.ca/easybuild/software/2023/x86-64-v3/MPI/gcc12/openmpi4/esmf/8.8.0/lib/python3.11/site-packages/esmpy/interface/loadESMF.py:94: VersionWarning: ESMF installation version 8.8.0, ESMPy version 8.8.0b0
  warnings.warn("ESMF installation version {}, ESMPy version {}".format(


In [ ]:
cat = xs.DataCatalog(CONFIG['extraction']['simulation']['search_data_catalogs']['data_catalogs'][0])

In [ ]:
ds_raw=cat.search(
    processing_level= 'raw',
    experiment= 'historical',
    source= 'CRCM5-SN',
    driving_model= 'MPI-ESM1-2-LR',
    driving_member= 'r1i1p1f1',
    variable='pr',
    member= 'r1').to_dataset()
# to make it work with figanos
ds_raw=ds_raw.rename({'crs':'rotated_pole'})
ds_raw

In [ ]:
def plot_martin(ds):
    
    # Sélectionner la région
    lon_bnds = [-116, -50]
    lat_bnds = [25, 70]
    
    data = xs.spatial.subset(ds.pr,method='bbox', name='QC', lon_bnds=lon_bnds, lat_bnds=lat_bnds)
    data=xc.core.units.convert_units_to(data, 'mm/day', context='hydro')
    

    dt = datetime.fromisoformat(str(data.time.values[0])[:26])
    date_clean = dt.strftime("%Y-%m-%d")

    # Créer une projection cartographique
    projection = ccrs.PlateCarree()

    # Initialiser la figure et les axes
    fig, ax = plt.subplots(figsize=(8, 6), subplot_kw={'projection': projection})

    # Afficher la variable
    p = ax.pcolormesh(data['lon'], data['lat'], data.squeeze(), norm=LogNorm(vmin=1, vmax=100), cmap='turbo')

    ax.set_xlim(-95, -50)
    ax.set_ylim(25, 70)

    # Ajouter les lignes de côtes
    ax.coastlines(resolution='10m', color='black', linewidth=1)
    plt.colorbar(p,orientation="vertical", label="(mm/d)", fraction=0.046, pad=0.04)


def fgplot_martin(ds):

    data = xs.spatial.subset(ds.pr,method='bbox', name='QC', lon_bnds=[-116, -50], lat_bnds=[25, 70])
    with xr.set_options(keep_attrs=True):
        data=xc.core.units.convert_units_to(data, 'mm/day', context='hydro')
    # fix xclim bug
    #data.attrs['long_name']= 'Precipitation'

    ax=fg.gridmap(data,
                  #ax=ax,
                  fig_kw=dict(figsize=(8, 6)),
                  plot_kw=dict(norm=LogNorm(vmin=1, vmax=100), cmap='turbo', 
                              cbar_kwargs=dict(orientation="vertical", fraction=0.046, pad=0.04)),
                  features=dict(coastline=dict(scale='10m', color='black', linewidth=1)),
                  projection=ccrs.PlateCarree(),
                  show_time=True,
                 )

    ax.set_xlim(-95, -50)
    ax.set_ylim(25, 70)

    return ax


plot_martin(ds_raw.sel(time='2001-03-03'))
fgplot_martin(ds_raw.sel(time='2001-03-03'))

In [ ]:
ds_adj= xr.open_zarr(f"{CONFIG['paths']['final']}/staging/simulation/biasadjusted/ESPO6_v20/CMIP6/CORDEX/NAM-C3/MPI-ESM1-2-LR/r1i1p1f1/OURANOS/CRCM5-SN/day/pr/pr_day_ESPO6_v20_CaSRv31+CMIP6_CORDEX_MPI-ESM1-2-LR_r1i1p1f1_OURANOS_CRCM5-SN_ssp370_r1_NAM-12_NAM-C3_1951-2100.zarr.zip")
ax=fgplot_martin(ds_adj.sel(time='2001-03-03'))
ax.set_title("LOESS f20n1d0tri")

ds_adj= xr.open_zarr(f"{CONFIG['paths']['final']}/loessf20n1d1tri/staging/simulation/biasadjusted/ESPO6_v20/CMIP6/CORDEX/NAM-C3/MPI-ESM1-2-LR/r1i1p1f1/OURANOS/CRCM5-SN/day/pr/pr_day_ESPO6_v20_CaSRv31+CMIP6_CORDEX_MPI-ESM1-2-LR_r1i1p1f1_OURANOS_CRCM5-SN_ssp370_r1_NAM-12_NAM-C3_1951-2100.zarr.zip")
ax=fgplot_martin(ds_adj.sel(time='2001-03-03'))
ax.set_title("LOESS loessf28n1d0tri ")


ds_adj= xr.open_zarr(f"{CONFIG['paths']['final']}/loessf28n1d0tri/staging/simulation/biasadjusted/ESPO6_v20/CMIP6/CORDEX/NAM-C3/MPI-ESM1-2-LR/r1i1p1f1/OURANOS/CRCM5-SN/day/pr/pr_day_ESPO6_v20_CaSRv31+CMIP6_CORDEX_MPI-ESM1-2-LR_r1i1p1f1_OURANOS_CRCM5-SN_ssp370_r1_NAM-12_NAM-C3_1951-2100.zarr.zip")
ax=fgplot_martin(ds_adj.sel(time='2001-03-03'))
ax.set_title("LOESS loessf28n1d0tri ")

ds_adj= xr.open_zarr(f"{CONFIG['paths']['final']}/loessf28n1d1tri/staging/simulation/biasadjusted/ESPO6_v20/CMIP6/CORDEX/NAM-C3/MPI-ESM1-2-LR/r1i1p1f1/OURANOS/CRCM5-SN/day/pr/pr_day_ESPO6_v20_CaSRv31+CMIP6_CORDEX_MPI-ESM1-2-LR_r1i1p1f1_OURANOS_CRCM5-SN_ssp370_r1_NAM-12_NAM-C3_1951-2100.zarr.zip")
ax=fgplot_martin(ds_adj.sel(time='2001-03-03'))
ax.set_title("LOESS loessf28n1d1tri ")

ds_adj= xr.open_zarr(f"{CONFIG['paths']['final']}/loessf20n1d1gau/staging/simulation/biasadjusted/ESPO6_v20/CMIP6/CORDEX/NAM-C3/MPI-ESM1-2-LR/r1i1p1f1/OURANOS/CRCM5-SN/day/pr/pr_day_ESPO6_v20_CaSRv31+CMIP6_CORDEX_MPI-ESM1-2-LR_r1i1p1f1_OURANOS_CRCM5-SN_ssp370_r1_NAM-12_NAM-C3_1951-2100.zarr.zip")
ax=fgplot_martin(ds_adj.sel(time='2001-03-03'))
ax.set_title("LOESS loessf20n1d1gau ")


ds_adj= xr.open_zarr(f"{CONFIG['paths']['final']}/loessf33n1d0tri/staging/simulation/biasadjusted/ESPO6_v20/CMIP6/CORDEX/NAM-C3/MPI-ESM1-2-LR/r1i1p1f1/OURANOS/CRCM5-SN/day/pr/pr_day_ESPO6_v20_CaSRv31+CMIP6_CORDEX_MPI-ESM1-2-LR_r1i1p1f1_OURANOS_CRCM5-SN_ssp370_r1_NAM-12_NAM-C3_1951-2100.zarr.zip")
ax=fgplot_martin(ds_adj.sel(time='2001-03-03'))
ax.set_title("LOESS loessf33n1d0tri ")